# Previsão de Tarifas de Corridas Uber
### Aprendizagem de Máquina — Desafio do Oráculo Preditivo — Grupo 03

Este notebook reúne todo o fluxo do projeto, na ordem em que deve ser executado:

1. Bibliotecas e configuração
2. Carga e diagnóstico dos dados
3. Limpeza
4. Engenharia de features
5. Análise exploratória (EDA)
6. Divisão treino/teste e padronização
7. Função de avaliação
8. Ajuste de hiperparâmetros (validação cruzada)
9. Experimento 1 — cada modelo com o máximo de dados viável
10. Experimento 2 — os três modelos nas mesmas 20.000 corridas
11. Validação cruzada dos dois experimentos
12. Gráficos e análise de erros
13. Salvamento dos modelos e tabelas usados pelo dashboard

**Tempo total de execução:** de 40 a 70 minutos, dependendo do computador. As células mais lentas (SVR) estão sinalizadas.

## 1. Bibliotecas e configuração

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import (r2_score, mean_absolute_error,
                             mean_squared_error, mean_absolute_percentage_error)

pd.set_option('display.float_format', '{:.4f}'.format)

PASTA_DADOS = Path('../data')
PASTA_MODELOS = Path('../app')
PASTA_MODELOS.mkdir(exist_ok=True)

AMOSTRA_COMUM = 20000   # tamanho do treino do SVR e do Experimento 2

## 2. Carga e diagnóstico dos dados

In [ ]:
df = pd.read_csv(PASTA_DADOS / 'Uber_DS.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
print("Tarifas acima de US$ 100:", (df['fare_amount'] > 100).sum())
print("Tarifas menores ou iguais a zero:", (df['fare_amount'] <= 0).sum())
print("\nPassageiros por corrida:")
print(df['passenger_count'].value_counts().sort_index())
print("\nLongitude de embarque igual a zero:", (df['pickup_longitude'] == 0).sum())
print("Latitude de embarque igual a zero:", (df['pickup_latitude'] == 0).sum())

fora_ny = (
    ~df['pickup_longitude'].between(-74.3, -73.7) | ~df['pickup_latitude'].between(40.5, 41.0) |
    ~df['dropoff_longitude'].between(-74.3, -73.7) | ~df['dropoff_latitude'].between(40.5, 41.0)
)
print("Corridas com alguma coordenada fora da região de Nova York:", fora_ny.sum())
print("Valores distintos na coluna key:", df['key'].nunique())

**Leitura:** o `isnull()` encontra só 1 valor ausente, mas o `describe()` revela valores impossíveis
(tarifa negativa, 208 passageiros, latitude 1644). As coordenadas iguais a zero são falha de GPS
gravada como zero, por isso não aparecem como nulas. A coluna `key` tem só 3.600 valores distintos:
é um identificador corrompido, sem relação com a tarifa.

## 3. Limpeza

In [ ]:
df_limpo = df.dropna()
df_limpo = df_limpo[
    (df_limpo['fare_amount'] > 0) &
    (df_limpo['fare_amount'] <= 100) &
    (df_limpo['passenger_count'].between(1, 6)) &
    (df_limpo['pickup_longitude'].between(-74.3, -73.7)) &
    (df_limpo['pickup_latitude'].between(40.5, 41.0)) &
    (df_limpo['dropoff_longitude'].between(-74.3, -73.7)) &
    (df_limpo['dropoff_latitude'].between(40.5, 41.0))
].copy()

print("Antes:", len(df), "| Depois:", len(df_limpo), "| Removidos:", len(df) - len(df_limpo))

**Por que não usar o IQR?** O limite superior pelo IQR seria Q3 + 1,5 × IQR = US$ 22,25. Esse corte
eliminaria todas as corridas de aeroporto (US$ 50 a 70), que são reais e formam um regime de preço
que o modelo precisa aprender. Usou-se um corte conservador de US$ 100.

In [ ]:
q1, q3 = df['fare_amount'].quantile([0.25, 0.75])
print(f"Limite superior pelo IQR: US$ {q3 + 1.5 * (q3 - q1):.2f}")

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    """Distância em km entre dois pontos da superfície da Terra."""
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df_limpo['distancia_km'] = haversine(df_limpo['pickup_longitude'], df_limpo['pickup_latitude'],
                                     df_limpo['dropoff_longitude'], df_limpo['dropoff_latitude'])

print("Distância exatamente zero:", (df_limpo['distancia_km'] == 0).sum())
print("Distância abaixo de 100 m:", (df_limpo['distancia_km'] < 0.1).sum())

df_limpo = df_limpo[df_limpo['distancia_km'] >= 0.1].copy()
print("Corridas restantes:", len(df_limpo),
      f"({(1 - len(df_limpo) / len(df)) * 100:.2f}% removidas no total)")

## 4. Engenharia de features

In [ ]:
df_limpo['pickup_datetime'] = pd.to_datetime(df_limpo['pickup_datetime'])
df_limpo['ano'] = df_limpo['pickup_datetime'].dt.year
df_limpo['mes'] = df_limpo['pickup_datetime'].dt.month
df_limpo['dia_semana'] = df_limpo['pickup_datetime'].dt.dayofweek
df_limpo['hora'] = df_limpo['pickup_datetime'].dt.hour

AEROPORTOS = {'JFK': (-73.7781, 40.6413), 'LGA': (-73.8740, 40.7769), 'EWR': (-74.1745, 40.6895)}

df_limpo['eh_aeroporto'] = 0
for nome, (lon, lat) in AEROPORTOS.items():
    perto = ((haversine(df_limpo['pickup_longitude'], df_limpo['pickup_latitude'], lon, lat) < 2) |
             (haversine(df_limpo['dropoff_longitude'], df_limpo['dropoff_latitude'], lon, lat) < 2))
    df_limpo.loc[perto, 'eh_aeroporto'] = 1

print(df_limpo['eh_aeroporto'].value_counts())

In [ ]:
regimes = df_limpo.groupby('eh_aeroporto').agg(
    tarifa_media=('fare_amount', 'mean'),
    distancia_media=('distancia_km', 'mean'))
regimes['custo_por_km'] = regimes['tarifa_media'] / regimes['distancia_media']
regimes.index = ['Não aeroporto', 'Aeroporto']
regimes

In [ ]:
df_final = df_limpo.drop(columns=['Unnamed: 0', 'key', 'pickup_datetime']).copy()
df_final.to_csv(PASTA_DADOS / 'uber_tratado.csv', index=False)
print(df_final.shape)
print(df_final.columns.tolist())

## 5. Análise exploratória (EDA)

In [ ]:
print(df_final['fare_amount'].describe())

plt.figure(figsize=(10, 4))
sns.histplot(df_final['fare_amount'], bins=60)
plt.title('Distribuição da tarifa (fare_amount)')
plt.xlabel('Tarifa (US$)')
plt.tight_layout()
plt.savefig(PASTA_DADOS / 'distribuicao_tarifa.png', dpi=150)
plt.show()

In [ ]:
matriz_corr = df_final.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlação')
plt.tight_layout()
plt.savefig(PASTA_DADOS / 'matriz_correlacao.png', dpi=150)
plt.show()

matriz_corr['fare_amount'].drop('fare_amount').sort_values(key=abs, ascending=False)

In [ ]:
tarifa_por_hora = df_final.groupby('hora')['fare_amount'].mean()

plt.figure(figsize=(10, 4))
tarifa_por_hora.plot(marker='o')
plt.title('Tarifa média por hora do dia')
plt.xlabel('Hora'); plt.ylabel('Tarifa média (US$)')
plt.tight_layout()
plt.savefig(PASTA_DADOS / 'tarifa_por_hora.png', dpi=150)
plt.show()

**Leitura:** a hora tem correlação de Pearson praticamente nula com a tarifa, mas a tarifa média
varia ao longo do dia (pico às 5h). A relação é cíclica, não linear, e a correlação de Pearson
só mede relações lineares.

In [ ]:
amostra = df_final.sample(5000, random_state=42)

plt.figure(figsize=(9, 6))
sns.scatterplot(data=amostra, x='distancia_km', y='fare_amount', hue='eh_aeroporto', s=10, alpha=0.5)
plt.title('Distância × tarifa (amostra de 5.000 corridas)')
plt.xlabel('Distância (km)'); plt.ylabel('Tarifa (US$)')
plt.tight_layout()
plt.savefig(PASTA_DADOS / 'distancia_tarifa.png', dpi=150)
plt.show()

In [ ]:
features = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
            'passenger_count', 'distancia_km', 'ano', 'mes', 'dia_semana', 'hora', 'eh_aeroporto']

vif = []
for col in features:
    outras = [c for c in features if c != col]
    r2 = LinearRegression().fit(df_final[outras], df_final[col]).score(df_final[outras], df_final[col])
    vif.append({'feature': col, 'VIF': 1 / (1 - r2)})

pd.DataFrame(vif).sort_values('VIF', ascending=False)

Todos os VIF ficaram abaixo de 2,5, longe do limite usual de 5. As 11 features foram mantidas.

## 6. Divisão treino/teste e padronização

In [ ]:
X = df_final.drop(columns=['fare_amount'])
y = df_final['fare_amount']

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_treino_esc = scaler.fit_transform(X_treino)   # aprende média e desvio só no treino
X_teste_esc = scaler.transform(X_teste)         # aplica no teste (evita data leakage)

# Amostra comum: usada pelo SVR e pelos três modelos no Experimento 2
X_treino_comum = X_treino_esc[:AMOSTRA_COMUM]
y_treino_comum = y_treino[:AMOSTRA_COMUM]

print("Treino:", X_treino.shape, "| Teste:", X_teste.shape, "| Amostra comum:", X_treino_comum.shape)

## 7. Função de avaliação

In [ ]:
def avaliar(nome, modelo, X_tr, y_tr, X_te=None, y_te=None):
    """Treina o modelo, mede os tempos e calcula as métricas no treino e no teste."""
    X_te = X_teste_esc if X_te is None else X_te
    y_te = y_teste if y_te is None else y_te

    inicio = time.time()
    modelo.fit(X_tr, y_tr)
    tempo_treino = time.time() - inicio

    inicio = time.time()
    pred_teste = modelo.predict(X_te)
    tempo_previsao = time.time() - inicio

    pred_treino = modelo.predict(X_tr)
    r2_treino = r2_score(y_tr, pred_treino)
    r2_teste = r2_score(y_te, pred_teste)

    resultado = {
        'Modelo': nome,
        'Corridas no treino': len(y_tr),
        'R2 treino': r2_treino,
        'R2 teste': r2_teste,
        'Diferença R2': r2_treino - r2_teste,
        'MAE': mean_absolute_error(y_te, pred_teste),
        'RMSE': np.sqrt(mean_squared_error(y_te, pred_teste)),
        'MAPE (%)': mean_absolute_percentage_error(y_te, pred_teste) * 100,
        'Tempo de treino (s)': tempo_treino,
        'Tempo de previsão (s)': tempo_previsao,
    }
    print(f"=== {nome} ===")
    print(f"R2 treino: {r2_treino:.4f} | R2 teste: {r2_teste:.4f} | MAE: {resultado['MAE']:.4f} | "
          f"RMSE: {resultado['RMSE']:.4f} | MAPE: {resultado['MAPE (%)']:.2f}% | "
          f"Treino: {tempo_treino:.1f} s | Previsão: {tempo_previsao:.2f} s")
    return resultado, pred_teste

## 8. Ajuste de hiperparâmetros

O ajuste é feito com **validação cruzada de 5 partições** sobre a **amostra comum de 20.000 corridas**,
para que os modelos sejam ajustados nas mesmas condições. O conjunto de teste não participa do ajuste.

### 8.1 Random Forest — GridSearchCV (cerca de 3 a 4 minutos)

In [ ]:
busca_rf = GridSearchCV(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    {'max_depth': [10, 20, None], 'min_samples_leaf': [1, 5, 10]},
    cv=5, scoring='r2'
)

inicio = time.time()
busca_rf.fit(X_treino_comum, y_treino_comum)
print(f"Tempo: {time.time() - inicio:.0f} s")

ajuste_rf = pd.DataFrame(busca_rf.cv_results_)[
    ['param_max_depth', 'param_min_samples_leaf', 'mean_test_score', 'std_test_score', 'rank_test_score']
].rename(columns={'param_max_depth': 'max_depth', 'param_min_samples_leaf': 'min_samples_leaf',
                  'mean_test_score': 'R2 médio', 'std_test_score': 'Desvio', 'rank_test_score': 'Posição'})
ajuste_rf['max_depth'] = ajuste_rf['max_depth'].astype(object).where(ajuste_rf['max_depth'].notna(), 'Sem limite')
ajuste_rf = ajuste_rf.sort_values('Posição')
ajuste_rf.to_csv(PASTA_DADOS / 'ajuste_rf.csv', index=False)
ajuste_rf

A configuração adotada (`max_depth=20`, `min_samples_leaf=5`) fica praticamente empatada com a melhor
(diferença na sexta casa decimal). Pelo princípio da parcimônia, manteve-se o limite de profundidade,
que gera árvores menores e com menor risco de overfitting.

### 8.2 SVR — comparação de kernels (cerca de 15 minutos)

As orientações pedem testar a robustez do SVR com kernels RBF e Linear. Os dois são treinados nas
mesmas condições (20.000 corridas, C = 100, epsilon = 0,1) e avaliados no conjunto de teste.

In [ ]:
kernels = []
res, _ = avaliar('SVR kernel Linear', SVR(kernel='linear', C=100, epsilon=0.1), X_treino_comum, y_treino_comum)
kernels.append(res)
res, _ = avaliar('SVR kernel RBF', SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1), X_treino_comum, y_treino_comum)
kernels.append(res)

kernels = pd.DataFrame(kernels)
kernels.to_csv(PASTA_DADOS / 'comparacao_kernels.csv', index=False)
kernels

### 8.3 SVR — ajuste de C e gamma (cerca de 20 a 30 minutos)

Cada combinação é avaliada com validação cruzada de 5 partições, e o resultado é impresso assim que
termina. O gamma 0,5 é testado só com C = 10: ele já mostra forte overfitting e é a opção mais lenta.
O C = 1000 é testado só com gamma 0,01: com gamma 'scale', aumentar o C de 10 para 100 já piorou o
resultado.

In [ ]:
combinacoes = [(10, 0.01), (10, 'scale'), (10, 0.5), (100, 0.01), (100, 'scale'), (1000, 0.01)]
ajuste_svr = []

for C, gamma in combinacoes:
    inicio = time.time()
    notas = cross_val_score(SVR(kernel='rbf', C=C, gamma=gamma, epsilon=0.1),
                            X_treino_comum, y_treino_comum, cv=5, scoring='r2', n_jobs=-1)
    tempo = time.time() - inicio
    ajuste_svr.append({'C': C, 'gamma': str(gamma), 'R2 médio': notas.mean(),
                       'Desvio': notas.std(), 'Tempo (s)': tempo})
    print(f"C={C:<5} gamma={str(gamma):<6} | R2 médio: {notas.mean():.4f} | "
          f"Desvio: {notas.std():.4f} | Tempo: {tempo:.0f} s")

ajuste_svr = pd.DataFrame(ajuste_svr).sort_values('R2 médio', ascending=False)
ajuste_svr.to_csv(PASTA_DADOS / 'ajuste_svr.csv', index=False)
ajuste_svr

In [ ]:
melhor = ajuste_svr.iloc[0]
SVR_C = int(melhor['C'])
SVR_GAMMA = float(melhor['gamma']) if melhor['gamma'] != 'scale' else 'scale'
print("Configuração escolhida para o SVR: C =", SVR_C, "| gamma =", SVR_GAMMA)

## 9. Experimento 1 — cada modelo com o máximo de dados viável

A Regressão Linear e o Random Forest usam as 153.388 corridas de treino. O SVR usa a amostra de
20.000, porque seu custo de treinamento cresce entre o quadrado e o cubo do número de observações.
Todos são avaliados nas mesmas 38.347 corridas de teste.

In [ ]:
modelo_dummy = DummyRegressor(strategy='mean')
modelo_lr = LinearRegression()
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=5,
                                  random_state=42, n_jobs=-1)
modelo_svr = SVR(kernel='rbf', C=SVR_C, gamma=SVR_GAMMA, epsilon=0.1)

exp1, previsoes = [], {}
res, previsoes['Dummy'] = avaliar('Dummy (média)', modelo_dummy, X_treino_esc, y_treino); exp1.append(res)
res, previsoes['Regressão Linear'] = avaliar('Regressão Linear', modelo_lr, X_treino_esc, y_treino); exp1.append(res)
res, previsoes['Random Forest'] = avaliar('Random Forest', modelo_rf, X_treino_esc, y_treino); exp1.append(res)
res, previsoes['SVR'] = avaliar('SVR (RBF ajustado)', modelo_svr, X_treino_comum, y_treino_comum); exp1.append(res)
res_svr = res

exp1 = pd.DataFrame(exp1)
exp1.to_csv(PASTA_DADOS / 'metricas_exp1.csv', index=False)
exp1

In [ ]:
coeficientes = pd.DataFrame({'feature': X.columns, 'coeficiente': modelo_lr.coef_}) \
    .sort_values('coeficiente', key=abs, ascending=False)
coeficientes.to_csv(PASTA_DADOS / 'coeficientes_lr.csv', index=False)
print(coeficientes.to_string(index=False))
print("\nIntercepto:", round(modelo_lr.intercept_, 2))

In [ ]:
importancias = pd.DataFrame({'feature': X.columns, 'importancia': modelo_rf.feature_importances_}) \
    .sort_values('importancia', ascending=False)
importancias.to_csv(PASTA_DADOS / 'importancias_rf.csv', index=False)
importancias

## 10. Experimento 2 — os três modelos nas mesmas 20.000 corridas

Comparação em condições equivalentes. O SVR é o mesmo do Experimento 1 (já treinado com esta amostra),
então seu resultado é reaproveitado.

In [ ]:
exp2 = []
res, _ = avaliar('Dummy (média)', DummyRegressor(strategy='mean'), X_treino_comum, y_treino_comum); exp2.append(res)
res, _ = avaliar('Regressão Linear', LinearRegression(), X_treino_comum, y_treino_comum); exp2.append(res)
res, _ = avaliar('Random Forest',
                 RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=5,
                                       random_state=42, n_jobs=-1),
                 X_treino_comum, y_treino_comum); exp2.append(res)
exp2.append(res_svr)

exp2 = pd.DataFrame(exp2)
exp2.to_csv(PASTA_DADOS / 'metricas_exp2.csv', index=False)
exp2

## 11. Validação cruzada (5 partições) dos dois experimentos (cerca de 8 minutos)

In [ ]:
def validar(experimento, nome, modelo, X_tr, y_tr):
    notas = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='r2', n_jobs=-1)
    print(f"{experimento} | {nome:<17} | partes: {notas.round(4)} | média: {notas.mean():.4f} | desvio: {notas.std():.4f}")
    return {'Experimento': experimento, 'Modelo': nome, 'Corridas': len(y_tr),
            **{f'Parte {i+1}': n for i, n in enumerate(notas)},
            'Média': notas.mean(), 'Desvio': notas.std()}

def rf_config():
    return RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=5, random_state=42)

cv = []
cv.append(validar('Experimento 1', 'Regressão Linear', LinearRegression(), X_treino_esc, y_treino))
cv.append(validar('Experimento 1', 'Random Forest', rf_config(), X_treino_esc, y_treino))
cv_svr = validar('Experimento 1', 'SVR', SVR(kernel='rbf', C=SVR_C, gamma=SVR_GAMMA, epsilon=0.1),
                 X_treino_comum, y_treino_comum)
cv.append(cv_svr)
cv.append(validar('Experimento 2', 'Regressão Linear', LinearRegression(), X_treino_comum, y_treino_comum))
cv.append(validar('Experimento 2', 'Random Forest', rf_config(), X_treino_comum, y_treino_comum))
cv.append({**cv_svr, 'Experimento': 'Experimento 2'})

cv = pd.DataFrame(cv)
cv.to_csv(PASTA_DADOS / 'validacao_cruzada.csv', index=False)
cv

**Leitura:** com 20.000 corridas, todos os modelos oscilam mais entre as partições (desvio em torno de 0,02),
inclusive a Regressão Linear. A variação se deve ao tamanho da amostra, e não a uma instabilidade
específica do SVR.

## 12. Gráficos e análise de erros (Experimento 1)

In [ ]:
modelos_grafico = [('Regressão Linear', previsoes['Regressão Linear']),
                   ('Random Forest', previsoes['Random Forest']),
                   ('SVR (RBF ajustado)', previsoes['SVR'])]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (nome, pred) in zip(axes, modelos_grafico):
    ax.scatter(y_teste, pred, alpha=0.2, s=8, edgecolors='none')
    ax.plot([0, 100], [0, 100], 'r--', linewidth=2)
    ax.set_title(f'{nome} — R² = {r2_score(y_teste, pred):.4f}')
    ax.set_xlabel('Valor real (US$)'); ax.set_ylabel('Valor predito (US$)')
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(PASTA_DADOS / 'real_vs_predito.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (nome, pred) in zip(axes, modelos_grafico):
    ax.scatter(pred, y_teste - pred, alpha=0.2, s=8, edgecolors='none')
    ax.axhline(0, color='red', linestyle='--', linewidth=2)
    ax.set_title(f'Resíduos — {nome}')
    ax.set_xlabel('Valor predito (US$)'); ax.set_ylabel('Resíduo (real − predito)')
    ax.set_xlim(0, 80); ax.set_ylim(-40, 40)
plt.tight_layout()
plt.savefig(PASTA_DADOS / 'residuos.png', dpi=150)
plt.show()

In [ ]:
faixas = [0, 10, 20, 30, 40, 60, 100]
nomes_faixas = ['até 10', '10 a 20', '20 a 30', '30 a 40', '40 a 60', 'acima de 60']

tabela = pd.DataFrame({'Faixa de preço real (US$)': pd.cut(y_teste, bins=faixas, labels=nomes_faixas)})
for nome, pred in modelos_grafico:
    tabela[nome] = np.abs(y_teste.values - pred)

erro_faixa = tabela.groupby('Faixa de preço real (US$)', observed=True).agg(
    Corridas=('Random Forest', 'size'),
    **{nome: (nome, 'mean') for nome, _ in modelos_grafico}
).reset_index()
erro_faixa.to_csv(PASTA_DADOS / 'erro_por_faixa.csv', index=False)
erro_faixa

**Leitura:** o erro cresce com o preço da corrida em todos os modelos (heterocedasticidade). O SVR erra
menos nas corridas de até US$ 10, que são a maioria; o Random Forest erra menos em todas as outras faixas.
Isso explica por que o SVR vence no MAPE e o Random Forest vence no MAE e no RMSE.

## 13. Salvamento dos modelos e das tabelas

In [ ]:
joblib.dump(modelo_rf, PASTA_MODELOS / 'modelo_rf.pkl', compress=3)
joblib.dump(modelo_lr, PASTA_MODELOS / 'modelo_lr.pkl')
joblib.dump(scaler, PASTA_MODELOS / 'scaler.pkl')

for arquivo in sorted(PASTA_MODELOS.glob('*.pkl')):
    print(f"{arquivo.name:<16} {arquivo.stat().st_size / 1024 / 1024:8.2f} MB")
print("\nTabelas salvas em", PASTA_DADOS.resolve())